# Biomarker Analysis Pipeline (v3)

1:1 line-matched ICI vs never-ICI cohorts with two propensity score models.

### Matching
- **1:1** — one control per ICI case, matched on (cancer_type, line_category)

### Propensity score models
- **embeddings_only** — elastic net CV LR on text embeddings
- **all_covariates** — elastic net CV LR on embeddings + demographics + cancer type + panel version + line

### Analysis tracks
- **Track 1** — ICI-only, prognostic: `S(t) ~ base_vars + line_dummies + marker`
  - **unweighted** + **ATE** (1/ps generalizability weights)
- **Track 2** — Full cohort, predictive interaction: `S(t) ~ base_vars + line_dummies + marker + ICI + marker×ICI`
  - **noIPTW** + **ATE** weights

### Stages
1. **Data regeneration** — `generate_all_non_text_covariates.py`
2. **Line-matched cohorts** — `build_line_matched_cohort.py`
3. **Propensity scores** — `ICI_LRs.py` (elastic net CV)
4. **IPTW datasets** — `generate_IPTW_df.py --ps_model {embeddings_only,all_covariates}`
5. **Cox models** — `run_IPTW_analysis.py --ps_model {embeddings_only,all_covariates}`
6. **Compile results** — aggregate significant hits across all specs

In [ ]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
PS_MODELS = ['embeddings_only', 'all_covariates']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Data Regeneration (SV/Fusion fix)

Re-run `generate_all_non_text_covariates.py` to ensure `complete_somatic_data_df.csv` includes correct SV/Fusion columns.

In [ ]:
run_and_stream('Data regeneration',
               [sys.executable, '../data_preprocessing/generate_all_non_text_covariates.py'])

## Stage 2: Line-Matched Cohort Construction

Build 1:1 and 1:k matched cohorts on (cancer_type, line_category).

In [ ]:
run_and_stream('Line-matched cohorts',
               [sys.executable, 'build_line_matched_cohort.py'])

## Stage 3: Propensity Score Generation

Train embeddings-only and all-covariates elastic net CV logistic regression propensity models.

In [ ]:
run_and_stream('Propensity scores',
               [sys.executable, 'ICI_LRs.py'])

## Stage 4: IPTW Dataset Generation

Build IPTW datasets for each {matching, ps_model} combination.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'IPTW dataset ({ps_model})',
                   [sys.executable, 'generate_IPTW_df.py', '--ps_model', ps_model])

## Stage 5: Cox Model Analysis

Run Track 1 (ICI-only, ATE generalizability-weighted + unweighted) and Track 2 (full-cohort interaction, ATE + unweighted) for each PS model.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'Cox models ({ps_model})',
                   [sys.executable, 'run_IPTW_analysis.py', '--ps_model', ps_model])

## Stage 6: Compile Results

Aggregate significant hits across all specifications and apply cross-scheme robustness filtering.

In [ ]:
import re
import numpy as np
import pandas as pd

OUTPUT_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
COMPILED_PATH = os.path.join(OUTPUT_PATH, 'compiled_results/')
os.makedirs(COMPILED_PATH, exist_ok=True)

MATCHINGS = ['1to1']

# Track 1: ICI-only (ATE generalizability-weighted + unweighted)
TRACK1_WEIGHTS = ['unweighted', 'ATE']
# Track 2: full-cohort interaction (ATE + unweighted)
TRACK2_WEIGHTS = ['ATE', 'noIPTW']

# Discover cancer types from result filenames
cancer_types = set()
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for fname in os.listdir(run_path):
            m = re.match(r'(.+)_track[12]_', fname)
            if m:
                cancer_types.add(m.group(1))
cancer_types = sorted(cancer_types)
print(f"Discovered cancer types: {cancer_types}")

# ================================================
# 1. Compile all significant hits
# ================================================

# --- Track 1 ---
t1_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for wt in TRACK1_WEIGHTS:
                fname = os.path.join(run_path, f'{ct}_track1_{wt}_ICI_only.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_marker' in df.columns:
                    hits = df.loc[df['significant_marker']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = wt
                    hits['cancer_type'] = ct
                    t1_rows.append(hits)

t1_compiled = pd.concat(t1_rows, ignore_index=True) if t1_rows else pd.DataFrame()
t1_compiled.to_csv(os.path.join(COMPILED_PATH, 'track1_all_significant_hits.csv'), index=False)
print(f"Track 1 (ICI-only): {len(t1_compiled)} significant hits")

# --- Track 2 ---
t2_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for wt in TRACK2_WEIGHTS:
                fname = os.path.join(run_path, f'{ct}_track2_{wt}_interaction.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_predictive' in df.columns:
                    hits = df.loc[df['significant_predictive']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = wt
                    hits['cancer_type'] = ct
                    t2_rows.append(hits)

t2_compiled = pd.concat(t2_rows, ignore_index=True) if t2_rows else pd.DataFrame()
t2_compiled.to_csv(os.path.join(COMPILED_PATH, 'track2_all_significant_hits.csv'), index=False)
print(f"Track 2 (interaction): {len(t2_compiled)} significant hits")

# ================================================
# 2. Matched cohort patient counts by cancer type
# ================================================
COHORT_PATH = os.path.join(OUTPUT_PATH, 'matched_cohorts/')
cohort_counts_rows = []

for matching in MATCHINGS:
    cohort_file = os.path.join(COHORT_PATH, f'matched_cohort_{matching}.csv')
    if not os.path.isfile(cohort_file):
        print(f"  Cohort file not found: {cohort_file}")
        continue
    cohort = pd.read_csv(cohort_file)
    for ct, grp in cohort.groupby('CANCER_TYPE'):
        n_ici = int(grp['PX_on_ICI'].sum())
        n_ctrl = len(grp) - n_ici
        cohort_counts_rows.append({
            'cancer_type': ct,
            'matching': matching,
            'n_ICI': n_ici,
            'n_control': n_ctrl,
            'n_total': len(grp),
        })

cohort_counts = pd.DataFrame(cohort_counts_rows)
cohort_counts.to_csv(os.path.join(COMPILED_PATH, 'cohort_patient_counts.csv'), index=False)
print(f"\nCohort patient counts ({len(cohort_counts)} cancer types):")
print(cohort_counts.to_string(index=False))

# ================================================
# 3. Diagnostics summary
# ================================================
diag_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for ct in cancer_types:
            ess_file = os.path.join(run_path, f'{ct}_diagnostics/',
                                    'effective_sample_sizes.csv')
            if os.path.isfile(ess_file):
                ess = pd.read_csv(ess_file)
                ess['matching'] = matching
                ess['ps_model'] = ps_model
                diag_rows.append(ess)

if diag_rows:
    diag_df = pd.concat(diag_rows, ignore_index=True)
    diag_df.to_csv(os.path.join(COMPILED_PATH, 'scheme_diagnostics_summary.csv'), index=False)
    print(f"\nDiagnostics summary saved ({len(diag_df)} rows)")

print(f"\nAll outputs saved to {COMPILED_PATH}")